<a href="https://colab.research.google.com/github/giyuubin/group-project/blob/main/code/final_static_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# 1. AES-256 지원을 위한 고성능 zip 라이브러리 하나만 정확히 설치
!pip install pyzipper

import pyzipper
import glob
import os

# 2. 코랩에 업로드된 .zip 파일 탐색
zip_files = glob.glob("*.zip")
if not zip_files:
    print("❌ 코랩에 업로드된 .zip 파일이 없습니다! 왼쪽 파일 창에 .zip 파일을 먼저 업로드해주세요.")
else:
    target_zip = zip_files[0]
    print(f"📦 [{target_zip}] 호환 모드로 암호화 해제 시작...")

    try:
        # 3. pyzipper를 이용하여 AES-256 암호화 구조 해독
        with pyzipper.AESZipFile(target_zip) as zf:
            # 암호 세팅 (MalwareBazaar 표준 패스워드)
            zf.setpassword(b'infected')
            # 현재 가상 경로에 압축 해제
            zf.extractall()
        print("✅ 압축 해제 성공!")

        # 4. 압축 해제 후 생성된 .exe 파일 찾기
        exe_files = glob.glob("*.exe")
        # 혹시 기존에 이미 변경된 파일이 있다면 리스트에서 제외
        exe_files = [f for f in exe_files if f != 'malware_test.exe']

        if exe_files:
            original_exe = exe_files[0]
            # 파일명을 malware_test.exe로 일관성 있게 변경
            os.rename(original_exe, "malware_test.exe")
            print(f"🎯 파일 이름 변경 완료: {original_exe} -> malware_test.exe")
        else:
            print("⚠️ 압축은 풀렸으나 폴더 내에서 .exe 파일을 식별할 수 없습니다. 왼쪽 파일 창을 확인해주세요.")

    except Exception as e:
        print(f"❌ 압축 해제 프로세스 실패: {e}")

📦 [c34af1f1f238747d6839ce6857138e97d722443c4e2a794c072c236228ceaa07.zip] 호환 모드로 암호화 해제 시작...
✅ 압축 해제 성공!
🎯 파일 이름 변경 완료: c34af1f1f238747d6839ce6857138e97d722443c4e2a794c072c236228ceaa07.exe -> malware_test.exe


In [28]:
# 필수 보안 라이브러리 검증 및 설치 (SHAP 제거됨)
!pip install pefile > /dev/null 2>&1
print("📦 필수 보안 라이브러리(pefile) 검증 및 설치 완료!")

import pefile
import pandas as pd
import numpy as np
import joblib
import warnings
import sys
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

print("🛡️ LightGBM 실전 멀티 지표 정적 스캐너 기동...\n")

# ==========================================
# 1. 파일 경로 세팅
# ==========================================
MODEL_PATH = 'lightgbm_final.pkl'
TARGET_EXE = 'malware_test.exe'

# ==========================================
# 2. 모델 로드 및 딕셔너리 박스 언패킹
# ==========================================
try:
    loaded_data = joblib.load(MODEL_PATH)

    if isinstance(loaded_data, dict):
        model = loaded_data['model']
        target_features = loaded_data['features']
    else:
        model = loaded_data
        if hasattr(model, 'feature_name_'):
            target_features = list(model.feature_name_())
        else:
            target_features = [
                '_cexit', 'SearchPathW', 'WritePrivateProfileStringW', 'HeapSetInformation',
                'IsDlgButtonChecked', 'VirtualAlloc', 'exit', 'Polyline', 'Sleep',
                'GetPrivateProfileStringW', 'VariantInit', 'ClosePrinter', 'VirtualFree',
                'FreeEnvironmentStringsA', 'LoadLibraryA', 'ExitProcess', 'GetModuleHandleA',
                'RtlUnwind', 'GetDC', 'VirtualProtect', 'InterlockedCompareExchange',
                'GetCurrentDirectoryA', 'lstrcatA', 'CreateThread', 'GetScrollPos',
                'GetComputerNameW', 'IsChild', 'GetFileSize', 'VarBstrCat', '_controlfp',
                'InterlockedExchange', 'SetEndOfFile', 'FreeLibrary', 'QueryPerformanceCounter',
                'NetWkstaGetInfo', 'free', 'RegCloseKey', 'GetModuleFileNameA', 'RaiseException',
                'GetStartupInfoA', 'GetCommandLineA', 'GetVersionExA', 'GetSystemTimeAsFileTime',
                'GetCurrentProcessId', 'GetCurrentThreadId', 'GetTickCount', 'QueryPerformanceFrequency',
                'SetUnhandledExceptionFilter', 'UnhandledExceptionFilter', 'TerminateProcess'
            ]

    target_features = [col for col in target_features if col != 'malware']
    print(f"✅ [성공] 최종 학습 모델 피처 {len(target_features)}개 목록 동기화 완료!")

except Exception as e:
    print(f"❌ 초기화 에러: {e}")
    sys.exit()

# ==========================================
# 3. PE 파일 정적 분석 및 API 추출 함수
# ==========================================
def extract_apis_from_exe(exe_path):
    print(f"🔍 [{exe_path}] 정적 분석 중... (실행하지 않으므로 안전합니다)")
    api_set = set()
    try:
        pe = pefile.PE(exe_path)
        if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                for imp in entry.imports:
                    if imp.name is not None:
                        api_set.add(imp.name.decode('utf-8', 'ignore'))
        return api_set
    except Exception as e:
        print(f"❌ PE 파일 파싱 에러: {e}")
        return set()

# ==========================================
# 4. 바이너리 인코딩 및 모델 추론
# ==========================================
extracted_apis = extract_apis_from_exe(TARGET_EXE)

vector_dict = {feature: (1 if feature in extracted_apis else 0) for feature in target_features}
df_input = pd.DataFrame([vector_dict], columns=target_features)

print(f"✅ 벡터 변환 완료: 발견된 타겟 피처 {sum(vector_dict.values())}개 / {len(target_features)}개")

THRESHOLD = 0.85
pred_prob = model.predict_proba(df_input)[0][1]

# 탐지된 API 중 최대 3개 추출 (웹 프론트엔드 전달용)
detected_features = [feat for feat, val in vector_dict.items() if val == 1]
top_3_features = detected_features[:3]

# ==========================================
# 5. 코랩 시연용 대시보드 렌더링
# ==========================================
def show_colab_dashboard(prob, threshold, features):
    prob_percent = prob * 100
    is_malware = prob >= threshold

    color = "#dc3545" if is_malware else "#28a745"
    border_color = color
    status_text = "🚨 [경고] 악성코드 탐지 (차단 격리 수행)" if is_malware else "✅ [안전] 정상 파일입니다"

    html_content = f"""
    <div style="font-family: 'Malgun Gothic', sans-serif; background-color: #f8f9fa; padding: 25px; border-radius: 12px; max-width: 650px; box-shadow: 0 5px 15px rgba(0,0,0,0.1); border: 2px solid {color};">
        <h2 style="color: {color}; margin-top: 0; text-align: center; border-bottom: 2px dashed #dee2e6; padding-bottom: 15px;">{status_text}</h2>

        <div style="position: relative; width: 100%; height: 35px; background-color: #e9ecef; border-radius: 20px; margin: 35px 0 10px 0; overflow: visible;">
            <div style="height: 100%; width: {prob_percent}%; background-color: {color}; border-radius: 20px; box-shadow: inset 0 -3px 0 rgba(0,0,0,0.15);"></div>

            <div style="position: absolute; top: -10px; left: {threshold * 100}%; width: 3px; height: 55px; background-color: #dc3545; z-index: 10;"></div>
            <div style="position: absolute; top: -35px; left: calc({threshold * 100}% - 45px); font-size: 13px; font-weight: 900; color: #dc3545; background: white; padding: 4px 8px; border-radius: 5px; border: 2px solid #dc3545; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                차단선 (85%)
            </div>
        </div>

        <div style="display: flex; justify-content: space-between; font-weight: bold; color: #6c757d; font-size: 15px;">
            <span>0% (안전)</span>
            <span style="font-size: 24px; color: #212529; font-weight: 900;">{prob_percent:.2f}%</span>
            <span>100% (위험)</span>
        </div>

        <div style="margin-top: 30px; background: white; padding: 15px; border-radius: 8px; border: 1px solid #dee2e6;">
            <h4 style="color: #1f4e79; margin: 0 0 10px 0;">🔍 50대 타겟 API 중 탐지된 내역</h4>
            <ul style="list-style-type: none; padding: 0; margin: 0;">
    """

    if features:
        for feat in features:
            html_content += f'<li style="background: #f8f9fa; margin: 8px 0; padding: 12px 15px; border-radius: 6px; border-left: 5px solid {border_color}; font-family: Courier, monospace; font-weight: bold; font-size: 14px;">✔️ {feat}</li>'
    else:
        html_content += f'<li style="background: #f8f9fa; margin: 8px 0; padding: 12px 15px; border-radius: 6px; border-left: 5px solid #6c757d; font-family: Courier, monospace; color: #6c757d;">탐지된 주요 API 타겟이 없습니다.</li>'

    html_content += """
            </ul>
        </div>
    </div>
    """
    display(HTML(html_content))

# 함수 실행
show_colab_dashboard(pred_prob, THRESHOLD, top_3_features)

📦 필수 보안 라이브러리(pefile) 검증 및 설치 완료!
🛡️ LightGBM 실전 멀티 지표 정적 스캐너 기동...

✅ [성공] 최종 학습 모델 피처 50개 목록 동기화 완료!
🔍 [malware_test.exe] 정적 분석 중... (실행하지 않으므로 안전합니다)
✅ 벡터 변환 완료: 발견된 타겟 피처 0개 / 50개
